In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


In [1]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import list_repo_files, hf_hub_download

REPO = "Luffy503/VoCo_Downstream"   # eva's source
files = list_repo_files(REPO, repo_type="dataset")
print([f for f in files if "ancreas" in f or "Panc" in f])

['Dataset082_TCIA_Pancreas-CT.zip', 'MSD_Decathlon/Dataset007_Pancreas.zip']


In [2]:
import os
from huggingface_hub import hf_hub_download

REPO = "Luffy503/VoCo_Downstream"
zip_path = hf_hub_download(
    REPO,
    filename="MSD_Decathlon/Dataset007_Pancreas.zip",
    repo_type="dataset",
    local_dir="/kaggle/temp/msd",
)
print(zip_path, round(os.path.getsize(zip_path) / 1e9, 2), "GB")

MSD_Decathlon/Dataset007_Pancreas.zip: reconstructing file:   0%|          |  0.00B / 12.2GB            

MSD_Decathlon/Dataset007_Pancreas.zip: downloading bytes:           |  0.00B            

/kaggle/temp/msd/MSD_Decathlon/Dataset007_Pancreas.zip 12.21 GB


In [3]:
import zipfile
z = zipfile.ZipFile(zip_path)
names = z.namelist()
print(len(names)); print(names[:8])
z.extractall("/kaggle/temp/msd/extracted")

705
['Dataset007_Pancreas/dataset.json', 'Dataset007_Pancreas/imagesTr/', 'Dataset007_Pancreas/imagesTr/pancreas_001_0000.nii.gz', 'Dataset007_Pancreas/imagesTr/pancreas_004_0000.nii.gz', 'Dataset007_Pancreas/imagesTr/pancreas_005_0000.nii.gz', 'Dataset007_Pancreas/imagesTr/pancreas_006_0000.nii.gz', 'Dataset007_Pancreas/imagesTr/pancreas_010_0000.nii.gz', 'Dataset007_Pancreas/imagesTr/pancreas_012_0000.nii.gz']


In [4]:
import zipfile
z = zipfile.ZipFile(zip_path)
wanted = [n for n in z.namelist()
          if n.startswith(("Dataset007_Pancreas/imagesTr/", "Dataset007_Pancreas/labelsTr/"))
          or n.endswith("dataset.json")]
print(len(wanted), "entries to extract")
for n in wanted:
    z.extract(n, "/kaggle/temp/msd/extracted")

import glob
imgs = glob.glob("/kaggle/temp/msd/extracted/Dataset007_Pancreas/imagesTr/*.nii.gz")
labs = glob.glob("/kaggle/temp/msd/extracted/Dataset007_Pancreas/labelsTr/*.nii.gz")
print(len(imgs), "images,", len(labs), "labels")   # expect 281 and 281

565 entries to extract
281 images, 281 labels


In [7]:
import json, numpy as np
from scipy import ndimage as ndi

OUT = "/kaggle/working/prep"
meta = json.load(open(f"{OUT}/index.json"))
rng = np.random.default_rng(42)

# --- splits (tumor cases only for now) ---
tumor_cases = [m["case"] for m in meta if m["tumor_vox"] > 0]
rng.shuffle(tumor_cases)
splits = {
    "test":        tumor_cases[:50],
    "val":         tumor_cases[50:65],
    "mask_train":  tumor_cases[65:115],     # keep masks (the "few masks" setting)
    "report_only": tumor_cases[115:265],    # masks hidden, reports only
}
json.dump(splits, open(f"{OUT}/splits.json", "w"), indent=1)
print({k: len(v) for k, v in splits.items()})

# --- clean tumor attributes from the mask ---
STRUCT = ndi.generate_binary_structure(3, 3)
MIN_VOX = 10   # ignore specks (< ~80 mm^3 at 2 mm voxels)

def clean_tumors(lab, spacing):
    cc, n = ndi.label(lab == 2, structure=STRUCT)
    out = []
    for i in range(1, n + 1):
        idx = np.argwhere(cc == i)
        if len(idx) < MIN_VOX:
            continue
        pts = idx * np.asarray(spacing)
        pts = pts - pts.mean(0)
        _, _, vt = np.linalg.svd(pts, full_matrices=False)   # principal axes
        proj = pts @ vt.T
        ext = np.sort(proj.max(0) - proj.min(0) + np.mean(spacing))[::-1]
        out.append(dict(d1=float(ext[0]), d2=float(ext[1])))
    out.sort(key=lambda t: -t["d1"])
    return out

# --- turn clean attributes into a (noisy) "report" ---
def make_report(clean, sigma, rng):
    tumors = []
    for t in clean:
        d1 = max(round(t["d1"] * np.exp(rng.normal(0, sigma))), 3)
        d2 = max(round(t["d2"] * np.exp(rng.normal(0, sigma))), 3)
        d2 = min(d1, d2)
        d3 = (d1 + d2) / 2                      # paper's rule when 2 diameters are given
        tumors.append(dict(d1_mm=d1, d2_mm=d2, vol_mm3=float(np.pi / 6 * d1 * d2 * d3)))
    return dict(n_tumors=len(tumors), tumors=tumors,
                total_vol_mm3=float(sum(t["vol_mm3"] for t in tumors)))

clean_all, reports = {}, {}
for c in splits["mask_train"] + splits["report_only"] + splits["val"] + splits["test"]:
    d = np.load(f"{OUT}/{c}.npz")
    clean_all[c] = clean_tumors(d["lab"], d["spacing"])
    reports[c] = make_report(clean_all[c], sigma=0.10, rng=rng)

json.dump(clean_all, open(f"{OUT}/clean_tumors.json", "w"))
json.dump(reports,   open(f"{OUT}/reports_sigma0.10.json", "w"))
print(reports[splits["report_only"][0]])

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/prep/index.json'